# 🧬 Pandas for Bioinformatics
## Data Manipulation & Cleaning in Computational Biology

---

**Course:** Computational Biology & Bioinformatics  
**Duration:** ~60 minutes  
**Level:** Beginner to Intermediate  
**Prerequisites:** Basic Python (variables, lists, loops)

---

### 🎯 What You Will Learn

By the end of this notebook, you will be able to:

1. Load, inspect, and understand biological datasets using Pandas
2. Clean messy real-world bioinformatics data (missing values, duplicates, wrong types)
3. Filter, sort, and query genomic and expression data
4. Group, aggregate, and summarize datasets by biological categories
5. Merge multiple datasets (like joining gene annotations to expression results)
6. Apply string operations to biological identifiers and sequences
7. Build a mini data-cleaning pipeline for an RNA-seq results table

---

### 🗺️ Notebook Map

| Section | Topic | Estimated Time |
|---------|-------|----------------|
| 1 | Why Pandas in Bioinformatics? | 3 min |
| 2 | DataFrames — The Core Data Structure | 8 min |
| 3 | Loading Biological Data | 7 min |
| 4 | Inspecting & Exploring Data | 8 min |
| 5 | Data Cleaning | 10 min |
| 6 | Filtering & Querying | 7 min |
| 7 | Grouping & Aggregation | 7 min |
| 8 | Merging Datasets | 5 min |
| 9 | String Operations on Biological Data | 5 min |
| 10 | Mini Pipeline: AMR Gene Table Cleanup | 10 min |

---

> **💡 How to use this notebook:** Run each cell in order using `Shift + Enter`. Read the explanations carefully before running — the *why* matters as much as the *how*.


---
## Section 1 — Why Pandas in Bioinformatics?

Bioinformatics produces **enormous, complex tabular data**:

- A single RNA-seq experiment produces a table of 20,000+ genes × dozens of samples
- A variant calling run produces thousands of rows describing mutations in a genome
- AMR screening databases list hundreds of resistance genes with metadata

You could open these in Excel — but:

| Problem | Excel | Pandas |
|---------|-------|--------|
| File size limit | ~1 million rows | Millions of rows |
| Reproducibility | Click-based, hard to repeat | Code-based, fully reproducible |
| Automation | Manual | Fully scriptable |
| Integration | Standalone | Integrates with Python ecosystem |

**Pandas** gives you a programmable spreadsheet that slots directly into your analysis pipeline.

### The mental model: a DataFrame is a smarter spreadsheet

```
        gene_id    gene_name    log2FC    p_value    padj
  0    GENE0001    mecA         3.45      0.0001     0.002
  1    GENE0002    blaZ         1.23      0.032      0.21
  2    GENE0003    tetA        -0.87      0.89       1.0
  ...  ...         ...          ...       ...        ...

  ↑ index (row numbers)    ↑ columns (features)
```

Each **row** is an observation (a gene, a sample, a variant).  
Each **column** is a variable (an attribute of that observation).


In [ ]:
# Install pandas and numpy
!pip install pandas numpy

In [37]:
# ============================================================
# 📦 Import everything we need for this notebook
# Run this cell first — it sets up our environment
# ============================================================

import pandas as pd
import numpy as np
import io

# Display settings for readable output
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")
print("✅ Environment ready!")

Pandas version: 2.2.2
NumPy version:  2.0.2
✅ Environment ready!


---
## Section 2 — DataFrames: The Core Data Structure

### 2.1 Creating a DataFrame from scratch

Before loading real files, let's build intuition by creating a small DataFrame by hand. We'll model a simple **gene expression comparison** between two bacterial conditions.

In [ ]:
# ============================================================
# Building a DataFrame from a dictionary
# Keys = column names | Values = lists of data
# ============================================================

gene_data = {
    'gene_id':    ['MRSA_0001', 'MRSA_0002', 'MRSA_0003', 'MRSA_0004', 'MRSA_0005'],
    'gene_name':  ['mecA',      'femA',      'femB',      'blaZ',      'pbp2a'    ],
    'function':   ['resistance','cell_wall', 'cell_wall', 'resistance','resistance'],
    'length_bp':  [2007,        1467,        1305,        858,         2067       ],
    'expression_control':  [12.3,  45.6,  33.2,  8.9,   15.1],
    'expression_treated':  [89.4,  44.1,  31.8,  67.2,  91.3],
}

df = pd.DataFrame(gene_data)
print("Our first DataFrame — a mini MRSA gene expression table:")


Our first DataFrame — a mini MRSA gene expression table:
     gene_id gene_name    function  length_bp  expression_control  expression_treated
0  MRSA_0001      mecA  resistance       2007             12.3000             89.4000
1  MRSA_0002      femA   cell_wall       1467             45.6000             44.1000
2  MRSA_0003      femB   cell_wall       1305             33.2000             31.8000
3  MRSA_0004      blaZ  resistance        858              8.9000             67.2000
4  MRSA_0005     pbp2a  resistance       2067             15.1000             91.3000


In [ ]:
df

,gene_id,gene_name,function,length_bp,expression_control,expression_treated
0,MRSA_0001,mecA,resistance,2007,12.3000,89.4000
1,MRSA_0002,femA,cell_wall,1467,45.6000,44.1000
2,MRSA_0003,femB,cell_wall,1305,33.2000,31.8000
3,MRSA_0004,blaZ,resistance,858,8.9000,67.2000
4,MRSA_0005,pbp2a,resistance,2067,15.1000,91.3000


### 2.2 Anatomy of a DataFrame

Let's explore the key attributes every DataFrame has:

In [ ]:
# Shape: (rows, columns)
print("Shape (rows x columns):", df.shape)
print()

# Column names
print("Column names:", df.columns.tolist())
print()

# Data types — critical in bioinformatics!
print("Data types per column:")
print(df.dtypes)
print()

# Index (row labels) — defaults to 0, 1, 2...
print("Row index:", df.index.tolist())

Shape (rows x columns): (5, 6)

Column names: ['gene_id', 'gene_name', 'function', 'length_bp', 'expression_control', 'expression_treated']

Data types per column:
gene_id                object
gene_name              object
function               object
length_bp               int64
expression_control    float64
expression_treated    float64
dtype: object

Row index: [0, 1, 2, 3, 4]


In [ ]:
# ============================================================
# Accessing data: columns and rows
# ============================================================

# Access a single column (returns a Series)
print("Gene names (single column):")
print(df['gene_name'])
print(df.gene_name)
print(type(df['gene_name']))  # It's a Series, not a DataFrame
print()



Gene names (single column):
0     mecA
1     femA
2     femB
3     blaZ
4    pbp2a
Name: gene_name, dtype: object
0     mecA
1     femA
2     femB
3     blaZ
4    pbp2a
Name: gene_name, dtype: object
<class 'pandas.core.series.Series'>



In [ ]:
# Access multiple columns (returns a DataFrame)
print("Gene name + function (multiple columns):")
df[['gene_name', 'function', 'length_bp']] # dataframe[["col1", "col2", "col3", ..."coln"]]

Gene name + function (multiple columns):


,gene_name,function,length_bp
0,mecA,resistance,2007
1,femA,cell_wall,1467
2,femB,cell_wall,1305
3,blaZ,resistance,858
4,pbp2a,resistance,2067


In [ ]:
# Access rows by position using .iloc (integer location)
print("First row (.iloc[0]):")
print(df.iloc[0])
print()

# Access rows by label using .loc
print("Rows 1 to 3 (.loc[1:3]):")
print(df.loc[1:3])

First row (.iloc[0]):
gene_id                MRSA_0001
gene_name                   mecA
function              resistance
length_bp                   2007
expression_control       12.3000
expression_treated       89.4000
Name: 0, dtype: object

Rows 1 to 3 (.loc[1:3]):
     gene_id gene_name    function  length_bp  expression_control  expression_treated
1  MRSA_0002      femA   cell_wall       1467             45.6000             44.1000
2  MRSA_0003      femB   cell_wall       1305             33.2000             31.8000
3  MRSA_0004      blaZ  resistance        858              8.9000             67.2000


In [ ]:
# ============================================================
# Creating new columns from existing ones
# (This is how you'd compute fold-change in a real analysis)
# ============================================================

df_copy = df.copy() # create a copy so as not to modify original dataset

# Fold change = treated / control
df_copy['fold_change'] = df_copy['expression_treated'] / df_copy['expression_control']

# Log2 fold change — standard in RNA-seq and expression analysis
df_copy['log2FC'] = np.log2(df['fold_change'])

# print("DataFrame with fold change and log2FC added:")
# df[['gene_name', 'expression_control', 'expression_treated', 'fold_change', 'log2FC']]
df_copy


# print("💡 Interpretation:")
# print("  log2FC > 0 → gene is UP-regulated in treated condition")
# print("  log2FC < 0 → gene is DOWN-regulated in treated condition")
# print("  log2FC ~ 0 → little change")

,gene_id,gene_name,function,length_bp,expression_control,expression_treated,fold_change,log2FC
0,MRSA_0001,mecA,resistance,2007,12.3000,89.4000,7.2683,2.8616
1,MRSA_0002,femA,cell_wall,1467,45.6000,44.1000,0.9671,-0.0483
2,MRSA_0003,femB,cell_wall,1305,33.2000,31.8000,0.9578,-0.0622
3,MRSA_0004,blaZ,resistance,858,8.9000,67.2000,7.5506,2.9166
4,MRSA_0005,pbp2a,resistance,2067,15.1000,91.3000,6.0464,2.5961


---
## Section 3 — Loading Biological Data

In practice, you will load data from files. Pandas handles many formats:

| Format | Function | Common in bioinformatics? |
|--------|----------|---------------------------|
| CSV | `pd.read_csv()` | Very common |
| TSV (tab-separated) | `pd.read_csv(sep='\t')` | Very common (DESeq2 output, BLAST, etc.) |
| Excel | `pd.read_excel()` | Common in lab metadata |
| JSON | `pd.read_json()` | API outputs, databases |
| Parquet | `pd.read_parquet()` | Large-scale genomics |

Since we're in a notebook, we'll simulate real files using string data — the functions work identically with actual files.

In [38]:
# ============================================================
# Simulated RNA-seq differential expression results
# (This mimics what DESeq2 or edgeR outputs as a TSV file)
# ============================================================

rnaseq_raw = """
gene_id\tgene_name\tbaseMean\tlog2FoldChange\tlfcSE\tstat\tpvalue\tpadj\tchromosome\tbiotype
GENE001\tmecA\t1245.3\t3.45\t0.21\t16.4\t0.000001\t0.00002\tchr1\tprotein_coding
GENE002\tblaZ\t892.1\t2.89\t0.18\t16.1\t0.000002\t0.00003\tchr1\tprotein_coding
GENE003\ttetM\t234.7\t-1.23\t0.35\t-3.5\t0.00045\t0.004\tchr2\tprotein_coding
GENE004\tvanA\t\t4.12\t0.29\t14.2\t0.000005\t0.00008\tchr3\tprotein_coding
GENE005\trpoB\t678.9\t0.12\t0.09\t1.3\t0.18\t0.45\tchr1\tprotein_coding
GENE006\t\t345.2\t-0.45\t0.11\t-4.1\t0.00004\t0.0008\tchr2\tprotein_coding
GENE007\tdfrA1\t1102.4\t2.15\t0.22\t9.8\t0.0000001\t0.000003\tchr4\tprotein_coding
GENE008\taacC1\t567.8\t1.67\t0.19\t8.8\t0.0000005\t0.00001\tchr4\tprotein_coding
GENE009\tsulI\t234.5\t0.05\t0.08\t0.6\t0.54\t\tchr3\tprotein_coding
GENE010\tgyrA\t890.2\t-2.34\t0.28\t-8.4\t0.0000003\t0.000005\tchr1\tprotein_coding
GENE011\tmecA\t1245.3\t3.45\t0.21\t16.4\t0.000001\t0.00002\tchr1\tprotein_coding
GENE012\tparC\t456.7\t1.98\t0.24\t8.3\t0.0000004\t0.000007\tchr2\tprotein_coding
GENE013\terma\t123.4\t0.78\t0.15\t5.2\t0.0000002\t0.000004\tchr5\trRNA
GENE014\tint1\t789.1\t2.45\t0.31\t7.9\t0.0000009\t0.00002\tchr6\tprotein_coding
GENE015\tblaTEM\t2341.0\t5.12\t0.41\t12.5\t0.0000001\t0.000002\tchr1\tprotein_coding
GENE015\tblaTEM\t2341.0\t5.12\t0.41\t12.5\t0.0000001\t0.000002\tchr1\tprotein_coding
""".strip()

# create csv file
with open('rna_seq_data.csv', 'w') as f:
    f.write(rnaseq_raw)

# Load from string — in real life: pd.read_csv('results.tsv', sep='\t')
df_rna = pd.read_csv('/content/rna_seq_data.csv', sep='\t')

print("RNA-seq results loaded! Shape:", df_rna.shape)
# print()
df_rna.head(3)

RNA-seq results loaded! Shape: (16, 10)


,gene_id,gene_name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,chromosome,biotype
0,GENE001,mecA,1245.3000,3.4500,0.2100,16.4000,0.0000,0.0000,chr1,protein_coding
1,GENE002,blaZ,892.1000,2.8900,0.1800,16.1000,0.0000,0.0000,chr1,protein_coding
2,GENE003,tetM,234.7000,-1.2300,0.3500,-3.5000,0.0004,0.0040,chr2,protein_coding


In [ ]:
# !wget("path/to/online/data/repository.csv")
# !curl

In [ ]:
df_rna.tail(3)

,gene_id,gene_name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,chromosome,biotype
12,GENE013,erma,123.4000,0.7800,0.1500,5.2000,0.0000,0.0000,chr5,rRNA
13,GENE014,int1,789.1000,2.4500,0.3100,7.9000,0.0000,0.0000,chr6,protein_coding
14,GENE015,blaTEM,2341.0000,5.1200,0.4100,12.5000,0.0000,0.0000,chr1,protein_coding


In [65]:
# ============================================================
# Simulated AMR gene annotation database
# (mimics CARD or ResFinder output)
# ============================================================

amr_db_raw = """
gene_name,drug_class,mechanism,identity_threshold,source_db
mecA,beta-lactam,target alteration,90,CARD
blaZ,beta-lactam,antibiotic inactivation,85,CARD
tetM,tetracycline,antibiotic target protection,95,ResFinder
vanA,glycopeptide,antibiotic target alteration,90,CARD
dfrA1,diaminopyrimidine,antibiotic target replacement,80,ResFinder
aacC1,aminoglycoside,antibiotic inactivation,85,CARD
sulI,sulfonamide,antibiotic target replacement,90,ResFinder
gyrA,fluoroquinolone,antibiotic target alteration,95,CARD
parC,fluoroquinolone,antibiotic target alteration,95,CARD
blaTEM,beta-lactam,antibiotic inactivation,85,CARD
int1,mobile_element,integron integrase,70,INTEGRALL
""".strip()

df_amr = pd.read_csv(io.StringIO(amr_db_raw))
print("AMR annotation database loaded! Shape:", df_amr.shape)
print()
print(df_amr)

AMR annotation database loaded! Shape: (11, 5)

   gene_name         drug_class                      mechanism  identity_threshold  source_db
0       mecA        beta-lactam              target alteration                  90       CARD
1       blaZ        beta-lactam        antibiotic inactivation                  85       CARD
2       tetM       tetracycline   antibiotic target protection                  95  ResFinder
3       vanA       glycopeptide   antibiotic target alteration                  90       CARD
4      dfrA1  diaminopyrimidine  antibiotic target replacement                  80  ResFinder
5      aacC1     aminoglycoside        antibiotic inactivation                  85       CARD
6       sulI        sulfonamide  antibiotic target replacement                  90  ResFinder
7       gyrA    fluoroquinolone   antibiotic target alteration                  95       CARD
8       parC    fluoroquinolone   antibiotic target alteration                  95       CARD
9     blaTEM

---
## Section 4 — Inspecting & Exploring Data

The first thing you do with any new dataset is **explore it** — before running any analysis. This is called **Exploratory Data Analysis (EDA)**. In bioinformatics, poor EDA leads to misinterpreted results.

### Your EDA checklist:
1. What is the shape? (How many genes/samples/variants?)
2. What are the columns and their types?
3. Are there missing values? Where?
4. Are there duplicates?
5. What is the distribution of key numeric columns?

In [ ]:
# ============================================================
# 4.1 First look functions — always run these first
# ============================================================

print("=" * 60)
print("HEAD — first 5 rows")
print("=" * 60)
print(df_rna.head())
print()

In [ ]:
print("=" * 60)
print("TAIL — last 5 rows")
print("=" * 60)
print(df_rna.tail())

In [ ]:
print("=" * 60)
print("INFO — column types + non-null counts")
print("=" * 60)
df_rna.info()

# print()
# print("💡 Notice:")
# print("  - 'baseMean' is float64 — good, it's a number")
# print("  - 'padj' has fewer non-null entries than other columns → missing values!")
# print("  - 'gene_name' is object (string) — expected")

INFO — column types + non-null counts
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   gene_id         15 non-null     object 
 1   gene_name       14 non-null     object 
 2   baseMean        14 non-null     float64
 3   log2FoldChange  15 non-null     float64
 4   lfcSE           15 non-null     float64
 5   stat            15 non-null     float64
 6   pvalue          15 non-null     float64
 7   padj            14 non-null     float64
 8   chromosome      15 non-null     object 
 9   biotype         15 non-null     object 
dtypes: float64(6), object(4)
memory usage: 1.3+ KB


In [ ]:
print("=" * 60)
print("DESCRIBE — summary statistics for numeric columns")
print("=" * 60)
print(df_rna.describe())

# print()
# print("💡 Key things to look for:")
# print("  - min/max: any implausible values?")
# print("  - count vs total rows: missing value indicator")
# print("  - mean vs median (50%): tells you about skewness")

DESCRIBE — summary statistics for numeric columns
       baseMean  log2FoldChange   lfcSE    stat  pvalue    padj
count   14.0000         15.0000 15.0000 15.0000 15.0000 14.0000
mean   796.1857          1.6140  0.2213  6.7667  0.0480  0.0325
std    579.7656          2.0946  0.0949  8.0254  0.1438  0.1202
min    123.4000         -2.3400  0.0800 -8.4000  0.0000  0.0000
25%    373.0750          0.0850  0.1650  0.9500  0.0000  0.0000
50%    734.0000          1.9800  0.2100  8.3000  0.0000  0.0000
75%   1049.8250          3.1700  0.2850 13.3500  0.0000  0.0001
max   2341.0000          5.1200  0.4100 16.4000  0.5400  0.4500


In [ ]:
# ============================================================
# 4.2 Detecting missing values — a critical EDA step
# ============================================================

print("Missing values per column:")
print("=" * 40)
missing = df_rna.isnull().sum() # isna().sum()
print(missing)

print()
print("Missing as % of total rows:")
print((missing / len(df_rna) * 100).round(1))

Missing values per column:
gene_id           0
gene_name         1
baseMean          1
log2FoldChange    0
lfcSE             0
stat              0
pvalue            0
padj              1
chromosome        0
biotype           0
dtype: int64

Missing as % of total rows:
gene_id          0.0000
gene_name        6.7000
baseMean         6.7000
log2FoldChange   0.0000
lfcSE            0.0000
stat             0.0000
pvalue           0.0000
padj             6.7000
chromosome       0.0000
biotype          0.0000
dtype: float64


In [ ]:
# ============================================================
# 4.3 Detecting duplicates
# ============================================================

print("Total duplicate rows:", df_rna.duplicated().sum())
print()

# Which rows are duplicated?
print("Duplicate rows:")
print(df_rna[df_rna.duplicated(keep=False)])

# print()
# print("⚠️  Notice: GENE001 and GENE011 are the same gene (mecA) — a true duplicate!")
# print("   This is common when merging datasets or when databases have redundant entries.")

Total duplicate rows: 1

Duplicate rows:
    gene_id gene_name  baseMean  log2FoldChange  lfcSE    stat  pvalue   padj chromosome         biotype
14  GENE015    blaTEM 2341.0000          5.1200 0.4100 12.5000  0.0000 0.0000       chr1  protein_coding
15  GENE015    blaTEM 2341.0000          5.1200 0.4100 12.5000  0.0000 0.0000       chr1  protein_coding


In [ ]:
# ============================================================
# 4.4 Value counts — useful for categorical columns
# ============================================================

print("Gene biotypes in our dataset:")
print(df_rna['biotype'].value_counts()) #df_rna.biotype.value_counts()
print()

print("Chromosomes represented:")
print(df_rna['chromosome'].value_counts())

Gene biotypes in our dataset:
biotype
protein_coding    14
rRNA               1
Name: count, dtype: int64

Chromosomes represented:
chromosome
chr1    6
chr2    3
chr3    2
chr4    2
chr5    1
chr6    1
Name: count, dtype: int64


---
## Section 5 — Data Cleaning

Real bioinformatics data is **messy**. Cleaning it is not optional — dirty data produces wrong biology.

### Common issues we'll fix:
1. Missing values (NaN) — decide: drop or fill?
2. Duplicate rows — remove true duplicates
3. Wrong data types — fix columns that should be numeric
4. Inconsistent strings — standardize gene names
5. Renaming and reorganizing columns

In [40]:
# ============================================================
# IMPORTANT: Always work on a copy — never modify the original
# This preserves the raw data in case you need to go back
# ============================================================

df_clean = df_rna.copy()
print("Working copy created. Shape:", df_clean.shape)

Working copy created. Shape: (16, 10)


In [41]:
# ============================================================
# 5.1 Remove duplicates
# ============================================================

print("Before removing duplicates:", df_clean.shape)

# Option A: Remove completely identical rows
df_clean = df_clean.drop_duplicates()
print("After removing identical rows:", df_clean.shape)

# Option B: Remove rows with the same gene_name (keep first occurrence)
# Use this when a gene appears multiple times in the input
df_clean = df_clean.drop_duplicates(subset=['gene_name'], keep='first')
print("After deduplicating on gene_name:", df_clean.shape)

# Reset index after removing rows (otherwise index has gaps: 0,1,2,4,5...)
df_clean = df_clean.reset_index(drop=True)
print()
print("Cleaned dataset (first 5 rows):")
print(df_clean.head())

Before removing duplicates: (16, 10)
After deduplicating on gene_name: (14, 10)

Cleaned dataset (first 5 rows):
   gene_id gene_name  baseMean  log2FoldChange  lfcSE    stat  pvalue   padj chromosome         biotype
0  GENE001      mecA 1245.3000          3.4500 0.2100 16.4000  0.0000 0.0000       chr1  protein_coding
1  GENE002      blaZ  892.1000          2.8900 0.1800 16.1000  0.0000 0.0000       chr1  protein_coding
2  GENE003      tetM  234.7000         -1.2300 0.3500 -3.5000  0.0004 0.0040       chr2  protein_coding
3  GENE004      vanA       NaN          4.1200 0.2900 14.2000  0.0000 0.0001       chr3  protein_coding
4  GENE005      rpoB  678.9000          0.1200 0.0900  1.3000  0.1800 0.4500       chr1  protein_coding


In [42]:
# ============================================================
# 5.2 Handling missing values
# Strategy depends on the biological meaning of each column
# ============================================================

print("Missing values before cleaning:")
print(df_clean.isnull().sum())
print()

Missing values before cleaning:
gene_id           0
gene_name         1
baseMean          1
log2FoldChange    0
lfcSE             0
stat              0
pvalue            0
padj              1
chromosome        0
biotype           0
dtype: int64



In [51]:
# --- Missing gene_name ---
# Strategy: fill with gene_id as a placeholder
# # Reasoning: we can't just drop the row — it might be biologically important
# mask = df_clean['gene_name'].isnull()
# df_clean.loc[mask, 'gene_name'] = df_clean.loc[mask, 'gene_id']

df_clean["gene_name"] = df_clean['gene_name'].fillna("UNK") # this would work if there is a missing value

# print("Genes where gene_name was filled from gene_id:")
# print(df_clean.loc[mask, ['gene_id', 'gene_name']])
# df_clean[df_clean.loc["gene_name"]]
# print
df_clean.gene_name
# print()

,gene_name
0,mecA
1,blaZ
2,tetM
3,vanA
4,rpoB
5,GENE006
6,dfrA1
7,aacC1
8,sulI
9,gyrA


In [52]:
# --- Missing baseMean ---
# baseMean = average count across samples (measure of expression level)
# Strategy: fill with the median of non-missing values
# Reasoning: median is robust to outliers; better than dropping the gene

median_baseMean = df_clean['baseMean'].median() # best method
# # using mode
# mode_baseMean = df_clean['baseMean'].mode()[0]
# # using preceding value
# prev_baseMean = df_clean['baseMean'].fillna(method='bfill')
# # using next value
# next_baseMean = df_clean['baseMean'].fillna(method='ffill')

print(f"Median baseMean: {median_baseMean:.2f}")

df_clean['baseMean'].fillna(median_baseMean, inplace=True)
print("baseMean missing values filled with median.")
print()

Median baseMean: 678.90
baseMean missing values filled with median.



/tmp/ipykernel_776/1595952536.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['baseMean'].fillna(median_baseMean, inplace=True)


In [53]:
df_clean["baseMean"]

,baseMean
0,1245.3000
1,892.1000
2,234.7000
3,678.9000
4,678.9000
5,345.2000
6,1102.4000
7,567.8000
8,234.5000
9,890.2000


In [54]:
# --- Missing padj (adjusted p-value) ---
# In DESeq2, padj is NA when a gene is flagged as outlier or low count
# Strategy: fill with 1.0 (non-significant) — this is the standard convention
# Reasoning: a gene without a valid test should be treated as non-significant

df_clean['padj'].fillna(1.0, inplace=True)
print("padj NAs filled with 1.0 (non-significant convention)")
print()

print("Missing values after cleaning:")
print(df_clean.isnull().sum())
print()
print("✅ No more missing values!")

padj NAs filled with 1.0 (non-significant convention)

Missing values after cleaning:
gene_id           0
gene_name         0
baseMean          0
log2FoldChange    0
lfcSE             0
stat              0
pvalue            0
padj              0
chromosome        0
biotype           0
dtype: int64

✅ No more missing values!


/tmp/ipykernel_776/3449980428.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['padj'].fillna(1.0, inplace=True)


In [55]:
# ============================================================
# 5.3 Renaming columns for clarity
# ============================================================

# Rename for readability
df_clean = df_clean.rename(columns={
    'log2FoldChange': 'log2FC',
    'lfcSE':          'log2FC_SE',
    'stat':           'Wald_stat',
    'pvalue':         'pval',
}) # df = df.rename(columns={"old column name": "new column name"})

print("Columns after renaming:")
print(df_clean.columns.tolist())

Columns after renaming:
['gene_id', 'gene_name', 'baseMean', 'log2FC', 'log2FC_SE', 'Wald_stat', 'pval', 'padj', 'chromosome', 'biotype']


In [56]:
# ============================================================
# 5.4 Adding a significance flag column
# Convention: padj < 0.05 AND |log2FC| > 1 = significant
# This is the standard cutoff in differential expression analysis
# ============================================================

df_clean['significant'] = (
    (df_clean['padj'] < 0.05) &
    (df_clean['log2FC'].abs() > 1.0)
)


df_clean['direction'] = 'unchanged'
df_clean.loc[(df_clean['significant']) & (df_clean['log2FC'] > 0), 'direction'] = 'up'
df_clean.loc[(df_clean['significant']) & (df_clean['log2FC'] < 0), 'direction'] = 'down'

print("Significant genes summary:")
print(df_clean['direction'].value_counts())
print()

print("Significant genes:")
print(df_clean[df_clean['significant']][['gene_name', 'log2FC', 'padj', 'direction']])

Significant genes summary:
direction
up           8
unchanged    4
down         2
Name: count, dtype: int64

Significant genes:
   gene_name  log2FC   padj direction
0       mecA  3.4500 0.0000        up
1       blaZ  2.8900 0.0000        up
2       tetM -1.2300 0.0040      down
3       vanA  4.1200 0.0001        up
6      dfrA1  2.1500 0.0000        up
7      aacC1  1.6700 0.0000        up
9       gyrA -2.3400 0.0000      down
10      parC  1.9800 0.0000        up
12      int1  2.4500 0.0000        up
13    blaTEM  5.1200 0.0000        up


In [58]:
print("=" * 60)
print("CLEANED DATASET SUMMARY")
print("=" * 60)
df_clean.head(10)

CLEANED DATASET SUMMARY


,gene_id,gene_name,baseMean,log2FC,log2FC_SE,Wald_stat,pval,padj,chromosome,biotype,significant,direction
0,GENE001,mecA,1245.3000,3.4500,0.2100,16.4000,0.0000,0.0000,chr1,protein_coding,True,up
1,GENE002,blaZ,892.1000,2.8900,0.1800,16.1000,0.0000,0.0000,chr1,protein_coding,True,up
2,GENE003,tetM,234.7000,-1.2300,0.3500,-3.5000,0.0004,0.0040,chr2,protein_coding,True,down
3,GENE004,vanA,678.9000,4.1200,0.2900,14.2000,0.0000,0.0001,chr3,protein_coding,True,up
4,GENE005,rpoB,678.9000,0.1200,0.0900,1.3000,0.1800,0.4500,chr1,protein_coding,False,unchanged
5,GENE006,GENE006,345.2000,-0.4500,0.1100,-4.1000,0.0000,0.0008,chr2,protein_coding,False,unchanged
6,GENE007,dfrA1,1102.4000,2.1500,0.2200,9.8000,0.0000,0.0000,chr4,protein_coding,True,up
7,GENE008,aacC1,567.8000,1.6700,0.1900,8.8000,0.0000,0.0000,chr4,protein_coding,True,up
8,GENE009,sulI,234.5000,0.0500,0.0800,0.6000,0.5400,1.0000,chr3,protein_coding,False,unchanged
9,GENE010,gyrA,890.2000,-2.3400,0.2800,-8.4000,0.0000,0.0000,chr1,protein_coding,True,down


---
## Section 6 — Filtering & Querying

Filtering lets you **ask biological questions** of your data:
- Which genes are significantly differentially expressed?
- Which genes are on chromosome 1?
- Which genes have a fold change > 4?

There are two main approaches: **boolean indexing** and **`.query()`**

In [59]:
# ============================================================
# 6.1 Boolean indexing (most common)
# Creates a True/False mask, then uses it to filter
# ============================================================

# Q: Which genes are significantly up-regulated?
upregulated = df_clean[df_clean['direction'] == 'up']
print(f"Up-regulated genes (n={len(upregulated)}):")
print(upregulated[['gene_name', 'log2FC', 'padj', 'chromosome']])
print()

Up-regulated genes (n=8):
   gene_name  log2FC   padj chromosome
0       mecA  3.4500 0.0000       chr1
1       blaZ  2.8900 0.0000       chr1
3       vanA  4.1200 0.0001       chr3
6      dfrA1  2.1500 0.0000       chr4
7      aacC1  1.6700 0.0000       chr4
10      parC  1.9800 0.0000       chr2
12      int1  2.4500 0.0000       chr6
13    blaTEM  5.1200 0.0000       chr1



In [60]:
# Q: Which genes on chr1 are significant?
# Multiple conditions: use & (AND), | (OR) — always wrap each condition in ()
chr1_sig = df_clean[
    (df_clean['chromosome'] == 'chr1') &
    (df_clean['significant'])
]
print(f"Significant genes on chr1 (n={len(chr1_sig)}):")
print(chr1_sig[['gene_name', 'chromosome', 'log2FC', 'padj']])
print()

Significant genes on chr1 (n=4):
   gene_name chromosome  log2FC   padj
0       mecA       chr1  3.4500 0.0000
1       blaZ       chr1  2.8900 0.0000
9       gyrA       chr1 -2.3400 0.0000
13    blaTEM       chr1  5.1200 0.0000



In [61]:
# Q: Filter for a list of genes of interest
# .isin() is perfect for this — like SQL's IN operator
amr_genes_of_interest = ['mecA', 'blaZ', 'blaTEM', 'vanA']

amr_subset = df_clean[df_clean['gene_name'].isin(amr_genes_of_interest)]
print(f"AMR genes of interest (n={len(amr_subset)}):")
print(amr_subset[['gene_name', 'log2FC', 'padj', 'direction']])
print()

AMR genes of interest (n=4):
   gene_name  log2FC   padj direction
0       mecA  3.4500 0.0000        up
1       blaZ  2.8900 0.0000        up
3       vanA  4.1200 0.0001        up
13    blaTEM  5.1200 0.0000        up



In [ ]:
# ============================================================
# 6.2 .query() method — more readable for complex filters
# ============================================================

# Same as above but more readable
result = df_clean.query("log2FC > 2 and padj < 0.05")
print("Genes with log2FC > 2 AND padj < 0.05:")
print(result[['gene_name', 'log2FC', 'padj']])
print()

In [ ]:
# ============================================================
# 6.3 Sorting — rank genes by significance or fold change
# ============================================================

# Sort by adjusted p-value (most significant first)
df_sorted_pval = df_clean.sort_values('padj', ascending=True)
print("Top 5 most significant genes:")
print(df_sorted_pval[['gene_name', 'log2FC', 'padj']].head())
print()

# Sort by log2FC absolute value (biggest effect size first)
df_sorted_fc = df_clean.sort_values('log2FC', key=abs, ascending=False)
print("Top 5 genes by effect size (|log2FC|):")
print(df_sorted_fc[['gene_name', 'log2FC', 'padj']].head())

---
## Section 7 — Grouping & Aggregation

**GroupBy** answers questions like:
- *How many up/down genes are on each chromosome?*
- *What is the average fold change per biotype?*
- *Which chromosome has the most significant AMR genes?*

The pattern is always: **Split → Apply → Combine**

```
df.groupby('column')['other_column'].aggregation_function()
```

In [62]:
# ============================================================
# 7.1 Counting genes per chromosome
# ============================================================

print("Gene count per chromosome:")
print(df_clean.groupby('chromosome').size())
print()

Gene count per chromosome:
chromosome
chr1    5
chr2    3
chr3    2
chr4    2
chr5    1
chr6    1
dtype: int64



In [63]:
# ============================================================
# 7.2 Multiple aggregations at once using .agg()
# ============================================================

chrom_stats = df_clean.groupby('chromosome').agg(
    gene_count    = ('gene_name', 'count'),
    mean_log2FC   = ('log2FC',    'mean'),
    max_log2FC    = ('log2FC',    'max'),
    sig_genes     = ('significant', 'sum'),   # sum of True = count of True
).round(3)

print("Per-chromosome summary:")
print(chrom_stats)

Per-chromosome summary:
            gene_count  mean_log2FC  max_log2FC  sig_genes
chromosome                                                
chr1                 5       1.8480      5.1200          4
chr2                 3       0.1000      1.9800          2
chr3                 2       2.0850      4.1200          1
chr4                 2       1.9100      2.1500          2
chr5                 1       0.7800      0.7800          0
chr6                 1       2.4500      2.4500          1


In [ ]:
# ============================================================
# 7.3 Cross-tabulation: chromosome × direction
# How many up/down/unchanged genes per chromosome?
# ============================================================

crosstab = pd.crosstab(df_clean['chromosome'], df_clean['direction'])
print("Gene regulation status by chromosome:")
print(crosstab)
print()
print("💡 chr1 and chr4 have the most up-regulated genes — interesting for follow-up!")

In [64]:
# ============================================================
# 7.4 GroupBy + transform: add group-level statistics as a new column
# Useful for normalization within groups
# ============================================================

# Add the mean baseMean of each chromosome as a new column
df_clean['chrom_mean_baseMean'] = df_clean.groupby('chromosome')['baseMean'].transform('mean')

print("First 6 rows with per-chromosome mean expression added:")
print(df_clean[['gene_name', 'chromosome', 'baseMean', 'chrom_mean_baseMean']].head(6))

First 6 rows with per-chromosome mean expression added:
  gene_name chromosome  baseMean  chrom_mean_baseMean
0      mecA       chr1 1245.3000            1209.5000
1      blaZ       chr1  892.1000            1209.5000
2      tetM       chr2  234.7000             345.5333
3      vanA       chr3  678.9000             456.7000
4      rpoB       chr1  678.9000            1209.5000
5   GENE006       chr2  345.2000             345.5333


---
## Section 8 — Merging Datasets

One of the most common tasks in bioinformatics is **joining two tables** — for example:
- RNA-seq results + gene annotations
- BLAST hits + metadata
- AMR screen results + resistance mechanism database

Pandas `merge()` works like SQL `JOIN`:

| Merge type | Description | Use case |
|------------|-------------|----------|
| `inner` | Only matching rows | Keep only annotated genes |
| `left` | All left rows + matching right | Keep all results, add annotation where available |
| `outer` | All rows from both | Full data inventory |
| `right` | All right rows + matching left | Keep all annotations |

**Most common in bioinformatics: `left` join** — you want all your results, with annotation where it exists.

In [66]:
# ============================================================
# 8.1 Left join: RNA-seq results + AMR annotation
# ============================================================

print("RNA-seq results (left table):", df_clean.shape)
print("AMR database (right table):  ", df_amr.shape)
print()

# Join on gene_name
df_merged = df_clean.merge(
    df_amr,
    on='gene_name',
    how='left'       # keep all RNA-seq results
)

print("Merged table shape:", df_merged.shape)
print()
print(df_merged[['gene_name', 'log2FC', 'padj', 'drug_class', 'mechanism']].head(12))

RNA-seq results (left table): (14, 13)
AMR database (right table):   (11, 5)

Merged table shape: (14, 17)

   gene_name  log2FC   padj         drug_class                      mechanism
0       mecA  3.4500 0.0000        beta-lactam              target alteration
1       blaZ  2.8900 0.0000        beta-lactam        antibiotic inactivation
2       tetM -1.2300 0.0040       tetracycline   antibiotic target protection
3       vanA  4.1200 0.0001       glycopeptide   antibiotic target alteration
4       rpoB  0.1200 0.4500                NaN                            NaN
5    GENE006 -0.4500 0.0008                NaN                            NaN
6      dfrA1  2.1500 0.0000  diaminopyrimidine  antibiotic target replacement
7      aacC1  1.6700 0.0000     aminoglycoside        antibiotic inactivation
8       sulI  0.0500 1.0000        sulfonamide  antibiotic target replacement
9       gyrA -2.3400 0.0000    fluoroquinolone   antibiotic target alteration
10      parC  1.9800 0.0000    flu

In [ ]:
# ============================================================
# 8.2 Analyzing merged data
# Which drug classes have the most up-regulated resistance genes?
# ============================================================

sig_amr = df_merged[
    (df_merged['significant']) &
    (df_merged['drug_class'].notna())
]

print(f"Significant genes with AMR annotation: {len(sig_amr)}")
print()

print("Significant AMR genes by drug class and direction:")
print(sig_amr.groupby(['drug_class', 'direction'])[['gene_name', 'log2FC']].agg(
    gene_count = ('gene_name', 'count'),
    mean_log2FC = ('log2FC', 'mean')
).round(2))

print()
print("💡 Biological interpretation:")
print("   Beta-lactam resistance genes are strongly up-regulated")
print("   — this strain may be under antibiotic selection pressure!")

In [ ]:
# ============================================================
# 8.3 Inner join: only genes present in BOTH tables
# ============================================================

df_inner = df_clean.merge(df_amr, on='gene_name', how='inner')
print(f"Inner join (only genes annotated in AMR DB): {df_inner.shape[0]} genes")
print(df_inner[['gene_name', 'drug_class', 'mechanism', 'log2FC', 'direction']])

---
## Section 9 — String Operations on Biological Data

Biological data is full of strings: gene IDs, sequence accessions, species names, functional descriptions. Pandas' `.str` accessor gives you powerful vectorized string operations — no loops needed.

In [67]:
# ============================================================
# String operations using .str accessor
# ============================================================

print("Gene names as uppercase:")
print(df_clean['gene_name'].str.upper()[:5].tolist())
print()

print("Gene IDs — numeric part only (.str.extract):")
print(df_clean['gene_id'].str.extract(r'(\d+)')[0][:5].tolist())
print()

print("Genes whose name contains 'bla' (beta-lactamase family):")
bla_genes = df_clean[df_clean['gene_name'].str.contains('bla', case=False, na=False)]
print(bla_genes[['gene_name', 'log2FC', 'direction']])
print()

Gene names as uppercase:
['MECA', 'BLAZ', 'TETM', 'VANA', 'RPOB']

Gene IDs — numeric part only (.str.extract):
['001', '002', '003', '004', '005']

Genes whose name contains 'bla' (beta-lactamase family):
   gene_name  log2FC direction
1       blaZ  2.8900        up
13    blaTEM  5.1200        up



In [ ]:
# ============================================================
# Working with biological sequences as strings
# ============================================================

# Create a small dataset of gene sequences
seq_data = pd.DataFrame({
    'gene':     ['mecA',   'blaZ',   'tetM',   'vanA'  ],
    'sequence': [
        'ATGAGTATTAATACTGAAAGAATGCAAATAGAATTTAAAGGAAATGA',
        'ATGAGTAAAATCAAAAGTTTTTTTATTTTAGTTTTGGCTTTTTTACC',
        'ATGAAATCTAAGAAAGAATTGATTAACGAAATAACAAAAGAACAA',
        'ATGAACAATGTTAATGAAATTATTGATAATGATGCAATGGTGATGA',
    ]
})

# Compute basic sequence properties
seq_data['length']   = seq_data['sequence'].str.len()
seq_data['GC_count'] = (seq_data['sequence'].str.count('G') +
                         seq_data['sequence'].str.count('C'))
seq_data['GC_percent'] = (seq_data['GC_count'] / seq_data['length'] * 100).round(1)
seq_data['starts_ATG'] = seq_data['sequence'].str.startswith('ATG')

print("Sequence analysis results:")
print(seq_data[['gene', 'length', 'GC_percent', 'starts_ATG']])
print()
print("💡 All start with ATG (Met start codon) — a good sanity check!")

---
## Section 10 — Mini Pipeline: AMR Gene Table Cleanup

Let's put it all together. This section simulates a **real task** you'd face in a bioinformatics job:

> *You receive the output of an AMR screening tool run against 8 bacterial isolates. The data is messy. You need to clean it, annotate it with resistance categories, and produce a tidy summary table for a public health report.*

This is the kind of task that separates a computational biologist from someone who just knows commands.

In [70]:
# ============================================================
# Input: raw AMR screening output (messy, real-world style)
# ============================================================

amr_screen_raw = """
sample_id,isolate_name,gene_hit,pct_identity,coverage_pct,resistance_class,contig
S001,MRSA_Ghana_01,mecA,99.5,100.0,beta-lactam,contig_14
S001,MRSA_Ghana_01,blaZ,87.3,98.2,beta-lactam,contig_02
S001,MRSA_Ghana_01,mecA,99.5,100.0,beta-lactam,contig_14
S002,MRSA_Ghana_02,mecA,98.1,100.0,beta-lactam,contig_07
S002,MRSA_Ghana_02,vanA,,95.3,glycopeptide,contig_11
S002,MRSA_Ghana_02,tetM,92.4,100.0,tetracycline,contig_03
S003,E_coli_KPC_01,blaTEM,95.2,99.8,beta-lactam,contig_01
S003,E_coli_KPC_01,blaTEM,95.2,99.8,beta-lactam,contig_01
S003,E_coli_KPC_01,sul1,88.7,97.3,sulfonamide,contig_05
S003,E_coli_KPC_01,dfrA1,91.3,100.0,trimethoprim,contig_05
S004,E_coli_KPC_02,blaTEM,96.1,100.0,beta-lactam,contig_02
S004,E_coli_KPC_02,aadA1,89.4,98.1,aminoglycoside,contig_03
S004,E_coli_KPC_02,aac3,78.2,62.5,aminoglycoside,contig_08
S005,Kleb_pneu_01,blaTEM,94.3,100.0,beta-lactam,contig_01
S005,Kleb_pneu_01,blaTEM,94.3,100.0,beta-lactam,contig_01
S005,Kleb_pneu_01,qnrS,83.6,95.4,quinolone,contig_06
S006,Kleb_pneu_02,blaOXA,91.2,99.0,beta-lactam,contig_04
S007,Salm_typhi_01,gyrA,96.8,100.0,fluoroquinolone,contig_02
S007,Salm_typhi_01,parC,94.1,98.7,fluoroquinolone,contig_02
S008,Salm_typhi_02,gyrA,97.2,100.0,fluoroquinolone,contig_01
""".strip()

df_amr_raw = pd.read_csv(io.StringIO(amr_screen_raw))
print("Raw AMR screen data loaded. Shape:", df_amr_raw.shape)
print()
print(df_amr_raw)

Raw AMR screen data loaded. Shape: (20, 7)

   sample_id   isolate_name gene_hit  pct_identity  coverage_pct resistance_class     contig
0       S001  MRSA_Ghana_01     mecA       99.5000      100.0000      beta-lactam  contig_14
1       S001  MRSA_Ghana_01     blaZ       87.3000       98.2000      beta-lactam  contig_02
2       S001  MRSA_Ghana_01     mecA       99.5000      100.0000      beta-lactam  contig_14
3       S002  MRSA_Ghana_02     mecA       98.1000      100.0000      beta-lactam  contig_07
4       S002  MRSA_Ghana_02     vanA           NaN       95.3000     glycopeptide  contig_11
5       S002  MRSA_Ghana_02     tetM       92.4000      100.0000     tetracycline  contig_03
6       S003  E_coli_KPC_01   blaTEM       95.2000       99.8000      beta-lactam  contig_01
7       S003  E_coli_KPC_01   blaTEM       95.2000       99.8000      beta-lactam  contig_01
8       S003  E_coli_KPC_01     sul1       88.7000       97.3000      sulfonamide  contig_05
9       S003  E_coli_KPC_0

In [71]:
# ============================================================
# Step 1: EDA — understand the problems before fixing them
# ============================================================

print("=" * 60)
print("STEP 1: Exploratory Data Analysis")
print("=" * 60)
print()

print("Missing values:")
print(df_amr_raw.isnull().sum())
print()

print("Duplicate rows:", df_amr_raw.duplicated().sum())
print()

print("Identity % range:")
print(df_amr_raw['pct_identity'].describe())
print()

print("Unique resistance classes:")
print(df_amr_raw['resistance_class'].unique())

STEP 1: Exploratory Data Analysis

Missing values:
sample_id           0
isolate_name        0
gene_hit            0
pct_identity        1
coverage_pct        0
resistance_class    0
contig              0
dtype: int64

Duplicate rows: 3

Identity % range:
count   19.0000
mean    92.7579
std      5.4934
min     78.2000
25%     90.3000
50%     94.3000
75%     96.4500
max     99.5000
Name: pct_identity, dtype: float64

Unique resistance classes:
['beta-lactam' 'glycopeptide' 'tetracycline' 'sulfonamide' 'trimethoprim'
 'aminoglycoside' 'quinolone' 'fluoroquinolone']


In [72]:
# ============================================================
# Step 2: Clean
# ============================================================

print("=" * 60)
print("STEP 2: Cleaning")
print("=" * 60)

df_pipe = df_amr_raw.copy()

# 2a. Remove exact duplicate hits (same gene on same contig)
n_before = len(df_pipe)
df_pipe = df_pipe.drop_duplicates(subset=['sample_id', 'gene_hit', 'contig'])
print(f"Removed {n_before - len(df_pipe)} duplicate rows. Now: {len(df_pipe)} rows")

# 2b. Filter: remove hits with low coverage (< 80%) — unreliable partial hits
n_before = len(df_pipe)
df_pipe = df_pipe[df_pipe['coverage_pct'] >= 80.0]
print(f"Removed {n_before - len(df_pipe)} low-coverage hits (<80%). Now: {len(df_pipe)} rows")

# 2c. Handle missing pct_identity
# Fill with the median — these are borderline hits that need review
median_id = df_pipe['pct_identity'].median()
n_missing = df_pipe['pct_identity'].isnull().sum()
df_pipe['pct_identity'].fillna(median_id, inplace=True)
print(f"Filled {n_missing} missing pct_identity values with median ({median_id:.1f}%)")

# 2d. Filter: remove hits below identity threshold (< 80%) — weak matches
n_before = len(df_pipe)
df_pipe = df_pipe[df_pipe['pct_identity'] >= 80.0]
print(f"Removed {n_before - len(df_pipe)} low-identity hits (<80%). Now: {len(df_pipe)} rows")

# 2e. Reset index
df_pipe = df_pipe.reset_index(drop=True)

print()
print("Cleaned data:")
print(df_pipe)

STEP 2: Cleaning
Removed 3 duplicate rows. Now: 17 rows
Removed 1 low-coverage hits (<80%). Now: 16 rows
Filled 1 missing pct_identity values with median (94.1%)
Removed 0 low-identity hits (<80%). Now: 16 rows

Cleaned data:
   sample_id   isolate_name gene_hit  pct_identity  coverage_pct resistance_class     contig
0       S001  MRSA_Ghana_01     mecA       99.5000      100.0000      beta-lactam  contig_14
1       S001  MRSA_Ghana_01     blaZ       87.3000       98.2000      beta-lactam  contig_02
2       S002  MRSA_Ghana_02     mecA       98.1000      100.0000      beta-lactam  contig_07
3       S002  MRSA_Ghana_02     vanA       94.1000       95.3000     glycopeptide  contig_11
4       S002  MRSA_Ghana_02     tetM       92.4000      100.0000     tetracycline  contig_03
5       S003  E_coli_KPC_01   blaTEM       95.2000       99.8000      beta-lactam  contig_01
6       S003  E_coli_KPC_01     sul1       88.7000       97.3000      sulfonamide  contig_05
7       S003  E_coli_KPC_01   

/tmp/ipykernel_776/2899504394.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_pipe['pct_identity'].fillna(median_id, inplace=True)


In [73]:
# ============================================================
# Step 3: Annotate — classify hits by confidence and resistance profile
# ============================================================

print("=" * 60)
print("STEP 3: Annotation")
print("=" * 60)

# Classify hit confidence by identity %
def classify_confidence(pct_id):
    if pct_id >= 95:
        return 'high'
    elif pct_id >= 85:
        return 'medium'
    else:
        return 'low'

df_pipe['hit_confidence'] = df_pipe['pct_identity'].apply(classify_confidence)

print("Hit confidence distribution:")
print(df_pipe['hit_confidence'].value_counts())
print()

# Flag genes of clinical priority (critical resistance genes)
critical_genes = {'mecA', 'vanA', 'blaTEM', 'blaOXA', 'blaKPC'}
df_pipe['clinical_priority'] = df_pipe['gene_hit'].isin(critical_genes)

print("Clinically critical resistance genes found:")
print(df_pipe[df_pipe['clinical_priority']][['sample_id', 'gene_hit', 'resistance_class', 'hit_confidence']])

STEP 3: Annotation
Hit confidence distribution:
hit_confidence
medium    9
high      6
low       1
Name: count, dtype: int64

Clinically critical resistance genes found:
   sample_id gene_hit resistance_class hit_confidence
0       S001     mecA      beta-lactam           high
2       S002     mecA      beta-lactam           high
3       S002     vanA     glycopeptide         medium
5       S003   blaTEM      beta-lactam           high
8       S004   blaTEM      beta-lactam           high
10      S005   blaTEM      beta-lactam         medium
12      S006   blaOXA      beta-lactam         medium


In [74]:
# ============================================================
# Step 4: Summarize — produce the report table
# ============================================================

print("=" * 60)
print("STEP 4: Summary Table for Public Health Report")
print("=" * 60)
print()

# Per-sample summary
summary = df_pipe.groupby('sample_id').agg(
    isolate         = ('isolate_name', 'first'),
    total_genes     = ('gene_hit', 'count'),
    unique_genes    = ('gene_hit', 'nunique'),
    drug_classes    = ('resistance_class', 'nunique'),
    critical_hits   = ('clinical_priority', 'sum'),
    high_conf_hits  = ('hit_confidence', lambda x: (x == 'high').sum()),
    mean_identity   = ('pct_identity', 'mean'),
).round(1)

# Add MDR flag: resistant to ≥ 3 drug classes
summary['MDR_flag'] = summary['drug_classes'] >= 3

print(summary.to_string())
print()
print("💡 Public Health Alert:")
mdr = summary[summary['MDR_flag'] == True]
print(f"   {len(mdr)} isolates meet Multi-Drug Resistant (MDR) criteria (≥3 drug classes):")
for _, row in mdr.iterrows():
    print(f"   - {row['isolate']} ({row['drug_classes']} drug classes, {int(row['critical_hits'])} critical genes)")

STEP 4: Summary Table for Public Health Report

                 isolate  total_genes  unique_genes  drug_classes  critical_hits  high_conf_hits  mean_identity  MDR_flag
sample_id                                                                                                                
S001       MRSA_Ghana_01            2             2             1              1               1        93.4000     False
S002       MRSA_Ghana_02            3             3             3              2               1        94.9000      True
S003       E_coli_KPC_01            3             3             3              1               1        91.7000      True
S004       E_coli_KPC_02            2             2             2              1               1        92.8000     False
S005        Kleb_pneu_01            2             2             2              1               0        88.9000     False
S006        Kleb_pneu_02            1             1             1              1               0  

In [76]:
# ============================================================
# Step 5: Export cleaned data
# In a real workflow, these save to disk:
#   df_pipe.to_csv('amr_cleaned.tsv', sep='\t', index=False)
#   summary.to_csv('amr_summary.csv')
#   summary.to_excel('amr_report.xlsx')
# ============================================================

# Simulate saving to TSV (showing what the file would look like)
# print("Final cleaned TSV output (first 8 rows):")
# df_pipe.to_csv("any_name_you_choose.csv", sep='\t', index=False)
df_pipe.to_csv("amr_clean.csv", index=False)

---
## 📝 Knowledge Check

Answer these questions to test your understanding. Try to answer before running code!

**Q1:** What function would you use to load a TSV file named `blast_results.txt`?

**Q2:** How would you find all rows in a DataFrame `df` where the column `evalue` is less than 0.001?

**Q3:** What does `df.groupby('species')['length'].mean()` compute?

**Q4:** After a `left` merge, some rows have `NaN` in the annotation columns. What does this mean biologically?

**Q5:** Why should you always `df.copy()` before cleaning, rather than modifying the original?

---

In [ ]:
# ============================================================
# 🏆 CHALLENGE EXERCISE
# ============================================================
#
# Using the VARIANT dataset below, complete the following tasks:
#
# 1. Load the data and inspect it (head, info, missing values)
# 2. Remove duplicate variants (same position + REF + ALT)
# 3. Fill missing QUAL with the median
# 4. Filter: keep only PASS variants with QUAL >= 20
# 5. Add a 'variant_type' column: 'SNP' if len(REF)==1 & len(ALT)==1, else 'INDEL'
# 6. Count how many SNPs and INDELs pass QC per chromosome
# 7. Export a summary table of variant counts per chromosome
#
# Hint: use .str.len() for step 5
# ============================================================

variant_raw = """
CHROM,POS,REF,ALT,QUAL,FILTER,DP,AF
chr1,145623,A,T,45.3,PASS,112,0.48
chr1,145623,A,T,45.3,PASS,112,0.48
chr1,289441,G,C,38.7,PASS,89,0.52
chr1,312098,AT,A,22.1,PASS,67,0.35
chr2,45219,C,T,,PASS,134,0.51
chr2,78934,G,GAATC,55.2,PASS,98,0.41
chr2,134567,T,A,12.4,LowQual,45,0.22
chr3,23415,A,G,61.8,PASS,156,0.49
chr3,89234,C,T,29.3,PASS,78,0.53
chr3,145098,G,A,8.9,LowQual,31,0.19
chr4,34521,A,T,48.2,PASS,118,0.47
chr4,67843,ATCG,A,33.6,PASS,92,0.39
chr4,98234,G,C,17.3,LowQual,52,0.28
""".strip()
with open('variants.csv', 'w') as f:
    f.write(variant_raw)

#--- YOUR CODE BELOW ---
# read the csv data
df_variants =

# Task 1: Inspect the data
print("Shape:", df_variants.shape)
print(df_variants.head())

# Complete the remaining tasks here...


---
## ✅ Notebook Summary

### What you learned:

| Concept | Key functions |
|---------|---------------|
| Create DataFrames | `pd.DataFrame()`, `pd.read_csv()` |
| Inspect data | `.head()`, `.info()`, `.describe()`, `.dtypes` |
| Find problems | `.isnull().sum()`, `.duplicated()`, `.value_counts()` |
| Clean data | `.drop_duplicates()`, `.fillna()`, `.rename()`, `.reset_index()` |
| Filter rows | Boolean indexing, `.isin()`, `.query()` |
| Sort | `.sort_values()` |
| Group & aggregate | `.groupby()`, `.agg()`, `pd.crosstab()` |
| Merge tables | `.merge(how='left'/'inner'/'outer')` |
| String operations | `.str.contains()`, `.str.upper()`, `.str.len()`, `.str.extract()` |
| Create new columns | `df['new_col'] = expression` |
| Apply functions | `.apply()` |

### The bioinformatics data pipeline pattern:

```
Load raw data → EDA → Clean → Annotate → Filter → Summarize → Export
```

This pattern applies whether you're working with:
- RNA-seq differential expression results
- Variant calling output (VCF)
- AMR screening results
- Metagenomic abundance tables
- Protein interaction data

---

### 🔗 Next steps

- **Visualization:** Use `matplotlib` and `seaborn` to make volcano plots, heatmaps, and bar charts from this data
- **Biopython:** Parse FASTA/GenBank files and bring sequences into Pandas
- **Real pipelines:** Connect this Pandas work to tools like BWA, GATK, or Salmon outputs
- **Practice datasets:** Try loading real NCBI datasets — download from [NCBI](https://www.ncbi.nlm.nih.gov/) or [ENA](https://www.ebi.ac.uk/ena)

---
*Built for the Computational Biology & Bioinformatics Program*